# GwenLand glcuda Wave 121 - LM-head tensor-core GEMM gate

Six counterbalanced T4 event-timed pairs with the production oracle.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave121-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "99c71b86c58b93dfacdae556193b1ec565ebaa92"
PATCH_SHA256 = "64581ab162e057dd3e6005667d3b1e372f842fde3d26212364851f5ea145147b"
PATCH_GZIP_B64 = """H4sIAPwEpmoC/8086XrcNpL/9RSwZlZhj9hU35ciT2Rb8WTjK5aSzK7jj2KToMQRjzYPHWPr+/Yh9gn3SbaqAJBgk91SvNn9Vj/U3QRQAKoKdYNe4Pus270IcuYcXIRu4TkH/NaJViHPDm6ca97vz+wgtldp4vIss7PcWQZhkN9ZacaWv3fETsxvmB+EnEWJx1m/15uMRjtB7PFb1nvkn2Uth0PuDYfuyPHGy+HA7btzPp1Mh31nPHJH7rw384dLz5/sdLtdduDx64O4CMOd/f39r1jxd9+xbs/ssf2+2R+P2Hff7ewfHDxhv8IwBuNYEHflOAafXuHmQRKzEgS7dNIYGi0aJsa+jcX+Q5OlRRzz1GTPf35xDOODyEnvmJvEOb/NTebEHnPCMHEdAuqEOU9jJ+csv+QCVMpzJ4i5R1097vM05V435VngFU7IVk5+mVnseRJFMF5bnx86FxlzUs6yYrUKA+4JeMs7hM1cmJWnbMn9BLqo/cGm0vxQdigygJ/dBLl7yYKM8VuA4gIXXfKUw2Z39ouMM0A2AFgseHwBq7Tz1AnyxeLzy/CEHpjshxiW/EO8KvL7w3II0Ic64ZfnSewHFyYTv8SwqiuubLF4SZ+i7RCnBgRmOXv3/u3rd2f2z29+OFuwvSxP2RHbfc2drEgRg7BojwNCoyAOsjxwWXaX5TwiMkarHLaYch/45s5iJ7A5QDO7TG5YnlxxILmTIopCIH/OLwBVkZOnwS2LijAPEBOCYrBKQBuwAFAo4lGS3pnsjMdZkgJNgEqCxH5wC+2hU8SAywueRDxPYdbdQ7WTn34+fn92cna6AIDBPznsY9wrG389fv/653en9ruT9zZ81fqUXU7+/u7k+dnJC1ui5OztjydvNGiD0Yjw5sewH6CFEXgZoOxDMRx87LDuU41M7PPOPoO/5hP8I+TYNBr+WXliX3PX6JhVj8i5tUEI2NQTuvW1NsD+iqdODvRZsJ7V05uSlX21YKP1ZyvsOB9rT1O+4k5ur3gMx+UOJrD0KSyrWvhiAQfGAYIZHdHhfmf/XqIBzqXtpJEhGgT7AkZ0LpRQNVTJJy7QNPDgmC7YMklC+TRJHTeER9ARnhBW3/MMZv/Wn4xM9iy5/da7Q8HhwXFJ0yRdLE7w4+lThWCxCivjub3kwCogK65sOvO278e2OvRGOX/nr4diZMhzlgCljhSMAJFglLTulD0DHzta4ghIIrEnRxs4iH35Qt1LslsOSE/gfm50cNQHsemPOouAxCrSmMHeDBAucGyeGFUj/u2KQSwKMmh1LxcgpqKjz/estih8oL4t/nqP8oe7OfeOPny+/7hr1kHCrkqksM9st/yxy2BkmNFDJUrh2drwBkZa2ksUrLfRZrSHHY0ZQXgkRqdzWLIffry9MsSEHBRlaEdZp2LLCFZo6LwDZ+sxrIMMEAEDOClI/SPZM75eLPCB0bGyq2Bl9Dsau5B+gq7YQVtxDHrJ0LeQXNlJauyCNrgA5t6mTNnrty9OXrEPzpflx90aa5ItIKaSE2iNKb/maYbtxA08e2Jgf+Q0D1SNjxg4BZFp7C53O/pAWABaGnYSu48ZLbsLGDUoqAyOdG1iCRljzGoIA/RerAroqQuJxQI05KXtkhYzdJWmn4mMc28h1jEa6NJy2zmXA3wH+FeNuVcrgpUAdwWAyr9qT8LE8WyirLFHHzUyAPPC6rEfzAqttnsJm9wTGNAlBB70kMfijG+QDL/rxCuNC3YPHJeceWng5wsGRxwm+HzfOM5q/rXn7Ut55NGrU9KVlKwMi8UC1FbJmW4Dv+6D+BWiwBaCWACopPBeTQxX3aHrX6qB2gZKgaM984M0az2ewlJSIC94jGoWbI44EUJ0t7aN7LLIveQmNqqjAITTRCiYQh+I70yWp0VdvmNPG3uAob5mmujdiDGkkt0DnjMZYsCsJjHlasuVaSKyJNfBAXu7zHh6TeZWN4nDO1BxYD2xVQIUpsVIS31usR85X7EEjG84v3CKoCMMu+YlKH2DQBmOggM2gvZuaQInYLGBsiFDPFQyhnADpzgEFVRCC/KaR3AFI0DVMJQHyAKwhtC5C+ILgj/o9boZeSSa6wD4scpDVxNnGiKFkFK6AtilDatIpVaE4ugc1k0mpzj9dRpZZavOV9QC2LJh7bHxJf/CSoW13ktyoKDCoKfWqs1axM61E4TOEqWvvjjwh+I8jBvi4gOpmUGvK3EC/Pf5t92ahv5td/H53vxt9zLJwBYs0YOPF9YcW2CnDzfkK9kywRZ5DmkGmuD+0aKpmmmtoUQDHOjcaelQCVsnY2Aqsr+g2wzWMTt4eHDT+jisn1RgtgvB5CUoepStn9UNtKjTg4YKasROxAFHv+1+vv9tF3CnFqghGp3MklDLu5xnoNscTz2JHDfbgGXSmTiXhdNsbNyAlKoDrWBja7Ukq4hvUmeFjNzrbOyPC97Ws9MUZEox1yTumt4Ee3CDoopRm6KRpgtj9Z8efWwaUh7217qUHyTJa/oHLCToDLJH2WCfKyCl3SwXca8bEvG1ClmsDQcbTbO4nd1yGPLip8JJcxqOukP5vOu6xVglWYCwNV3RwTG0XtLrdgDiGaxaHhcRKTo4PGvs/CipuV0XKTAwSmzzd7kYh7/3ePVnUkGI81XhWD9lEoXqCClUqd+wVL13q8RslYlfLQ9JgpVLbWmUK25pKencbIJ9tM2zQfZulb8PiditAxvidcMxlx8tRK6IC0fEC9y8S+bgI2hcEkxwRWavwD8QBJZ0bCGuWLCEdoAA6yRrJZU6iS2PAFMDfXgLBbSuo+2Pdd+lIRCFFERP2GuJWGepeyANrAMwvGux6WabDDgvByN/vpw77qQ/Gc57S28568EPfzgcLKf94ZLPvLk788eWNZnNRnPf9fqzqeOOZv5wPOCe6074dAxPRsP5uMdH/Z6jAtoYd96ytnokuqUdY84Tc8L2J+YUI84MQ7Nk+AGRUQreOKnHVg4YpAaIRHwIqirvWDtsB2OOwsH3/WCxcO3rJPBkwJQeZ3exC45/nkQBfH4+pi/PMFDF3qIIBZMUpJMEpAK4L0OKKOBzXNxgMIaV7Q8GYoFsVSwBeAqmLhjZuJVTjnFBod94FNlxf0JGFaeAjgiLMWEoHygLvb8QEWzGHfcS9Erc9QM0sr///g0rI9oo38lixkABAwuap99kFSgnRH191/ULVFJOnoORj3ro/evT/Z9mhDKLvV3hMQO7Mw9C3UQ/Pnhm7XQRVJu3rcXy2po1PK7vbHyMEfL1UD1GHg9FHBooCt4FbStNbqSrwJaOe6XBurkMRKj9Btxy9EJgucy5AJgZ6EywBjiH4yf6A5TMJqieWjiRbT40+32g23xkDuYPEc63AX+wRYT16WqEEZ0lX8h+sLB7YAaKlfIsCa95WyhSKncghv1pZl+ERT0myuPrIE3iCIhk8xgdAK/WXoU/EhCPaeDBcCAe0Opb7PVUBVLxh9Lw+mRsb68FhmajtSxARttYAFK1iRMkxL/+cEak4tGSex4Q9N3Z34moEhEMzZ2aHwosB+5hZE/H+5S6KUHllAvo4hnDKFgB7h8QGMMIADZPEmacv3yFuSH7zVv79evjo/45S1bgWIIHaSLbVKCQG8q9olupWAOXinP3mLcaOezlyevXHckmSH0gIM5nUNaF7T2HDz3AWGIAUzAiqqjbUqc89BcLintQoEvwgMI0ATXZG/C5Ow3P/YC9gmG0PMnwWQGGGznIzGGxk+JZKLcknHsFufTsK2ggQCx2jnOdIxLReu9Cry5+UT68Ouka3YFCYeDeSUf7Tx+8xDUuA6Br3Pko1XaFpdZNVtjQcKipuua52MzOpAcfRH/N/pR8c0SzlyEoeGQAX6LFWuudobFKdMHwU2IBZ0TOP5LUZPVnAdj1nbWxUeQISxeAPAUoU5ONO3jGqnjytQOIyYzdGtfudqwgs0GgC0McJdFwNjNnbH/Um6JAgiftx41YRprPOzXTC+ms9TkUcluttIlydlT1aBMTG7fw4uT7k/c26CD7/cnpDy9+Pn4ltpNh8LWzFkppm/dh+di2pjWj8iuXtwZlCyfWYwQ1smvKpJY7WFvJ8dkZLOLtr6d1/Kxz0MiutPIRPqg6b6UDgQdmGj0EPuUXn/Q5iN9G/TkaLKNRXxgsm9lNN1nMtseYv2vtXRo4Zp3Vmlh/kCy6OSECz80+nbVlaGRaX+AnZ4rB/AAzV7WWOj2EkdCf9keIq/50ODH7s83YIgPnkoNoTTVrTcX3yDTLWOTcSasO+gVp3ZhzPI8MOk0hKsuu3XSrqa0WbpY63NjLQC81TAORagl9qzmS5Kah7N/F4j0PnVsyBhpa61TUPCijFTRUfsM56Hewqy9BqQT/hHNSqi2GsQjyqDKrhFEBe1fpJErVp/JHjDYElWFkZCJa7AwwUxadyKISBh5SxpwKXAaLB10gqkoOKAVxIKtJyEKhegQyOdE0AEuZJb6KbXcvwmTphBUwXUlGRU6beFhNPpieJtqYrGbvdfQTuIFEa3HlLAejyaC+67Jcgq78mYqeUmCg6VryXInZwRSMD493L0AgSvON3AhZFOOlwTV6Gyxx3WLlxO4d+1SApWcJ23o6nKLLNphOe2Z/rIxrtI+TFbdzXFKGwStZdmGyS8C/7QVR+cBP+Sd76WRgFvjDAbGv8Qt3v4UfT032C+g7oCh5epnCF3l1BTiAi8Vf0DmT1ME+kiywAJWVVfVBTelvg0RVFTw2xvUBpUBt+wKzRPXoGRxDnuZPjCdb1FotvEjWn64oFYQHAYj/Xz9eLoDSpPik0wrmySPXoWVbfzecWny2ZT3kcej8iC5FEGFsPioytCvcEIu1+K3j5qFgR82/yCrhXIGRy8nYf/3Hf4Ikoey3yKyWIgvBWKv8VpTkoYAg2SDMR83zlCkr9DDJs2Av3/1sbQzEhMGyEYARz2TgZT71+Xw5c/ypPx8NhxPeH4xGfDnozcfLnjcbT0eTwYy7Y8sa9/rL3mQw9AZjpz+d9fh85o5crzcf+v3J3JsPxsv5fDkZbwy8yHkbARf5HA/uuI/HFv73h3ho6URRSA3O1OdT8Q2IJr6I3D3GRzTPeS2tX+Lte6rrev/mJSX4yWXBzCNK/GAZymkAoYfKbTlieRDxLvbmnqbsRH2AdBWKyUh5CjjJswccJNRR32RlbWBdAWN9YJtakuuhLGl6LUMTG/wnlbUXKgoWjsVvWE+4Ft6ptBZpoXM8BueYUF0m6PClUUZZVDfht0FGWQBM1UrlVEFTzCg0EnBjXX3VlgYmZEBy9wG1tb3Mou6iCd2BKxHPEYs87VKclRQ2hm0AXxiHQiceTILi4pJ9OK+XhYCKPv8oNMd8jgzY79WsLb13QznKcCFWsVANS6XUKiuFktEipKQCTxpHEOXbTZfKYKmgySx4SdyKppWhUxW6/u9YCPWip7ZgBLoAKs1+RJhasxlk43oeW2GxJXFto+9pfPmi5lssBEmwpAIJBIYvsmqQB06I2NtVhSW1VJGi14M7VzEobawKfDfUxAkV6bBzNO7OWS4rOaiKh0IoSDAZSSJDEIQA9YBlgjwD0xsOT93wfvb2VHSx2LHKyDPHB34g9lWsW1W5nH8U1cAphlIjTWOIpCAqKjkjHPoVSEvuRCZbwun1gdvyLo9hsah2VDU2QHNydpmEnrCpwNfCMHMpnOXRkMcCRdymcyKLMFAcZiAoALTDVsGKh3QSbmBxzA0TOKmgMbTI6h07B7NHrPS8AY2iuKskzWUUF/uTxqRMMzYjO7hJtKKKXxmBa4CBbYO8wN0r8bAqcqvuNFJhhsxfk9eNSXlLJjYjZ2V8yb6wrExxd6ysiGrhCDnXu1Q5XzAt7Bg9BFTfGKTEwORpEUWk071/OC7KTJn+D0q/QgMGgtsvQvTN0gCoSF7LJXevBE0DrJmOEQNlfQ27cFaCwkHegAZ9roOkyKhmx02KFCb2yA3CtC0iVaGyrLPpBrEfVtpFA4aFHoKsyIniBGgKS8fz/hY8r5pJPoljLAuJspbmMppcyopHEkuPXuDfu0sw/d+JnbLPEopZre5ei3d1ykzMiAIbg+nI7Pced0RkdD8muWXsYblb6S63S9gatkjsHbHnVKVPOQEqguu2S+HSMBVyw9jD4Y0SpI39Hwi44h9BXA+XoZYURZfWNoGrR73WI6UBrGavDIpqjaj6Yy5uQxQrFtJFhjC4UhcSQNB+yILI+0hNC4Y1KnQAQUN6N8DONVCkiLGsDJz3kFztIOuKUjJkXDiH10GG5uJmg5tkccPkVk+l0T0d9yez5dRfjuej0ay3nIwGs/HSncxmy/HQH/HJcu45y/nMskYeGN1gX7te35nOvInvz3uT/nLY830YueTj+Xg46nnzjUZ3OXPD7C5byPAe9sjyhg+KyQFHXqdOZBPDG3igF2zvb/DxWtyVubq2XWflgIt6J11mYlmwhxWLItnAWUY++hLLPl8YaOaL2C5WhhFjtpmy/DBIHT+gAU0rbV1P5RDBby9QP4lsTrFCdmRCM6BoOn71w8s3cIS84JpYKbyzqkXcODFoelFctSVk+uz07PjlSUs8uYQRRhQleBjUq9d/Ozl+YWN2pyVASsFRB+tOv0QKqU5ultUNAr6wtL40EgwgIKgWCuO11o1Vw1YtBhv4tZ3v7a1NgE+qOmzrxmS4kl95cHEJ5/2nmd07TRwQfZYFEk5mCsZjZI/98bj/R3FJdX8Elfb+EbGMcS0qF/Tt3Cu7utvsjhNb4IeARrHR2pGj2T5u0NjTmmUcQpH268E0GUIttg7zx+vnDqD4BQcuwQqo0HCt2BbxWJPhd8ARQqFfKgpVw1xHOxy/JulVBo+51TaZa8HYDpyr4SF2vjXZbYx8lfyDqDcb9DHPMxsMzHmlnVYFEWudIPctyNeurygcARsWKxtml+lF0mDLAjyGvXV01hSTDK1ghcuyCPGiHS6zkrNwdn4RTrQROllZH0Yp+X0MwcCaO2YNHslt1e+ZqH9kXiEugIEVkhRhadGwX94fvy7dV0fe99LMkdJVHvQxyUu+JSzmIlAOOPjDXVyMmrB1JzV4lPtF9aPWJmJKjoC3pKPoaYMzi72gfYIJh3dNMrYGEHEkbP5ql4A6F6QiGsWEPbQ5yN5UVwLxbk4aoEduNbRoBKwa4L6ADcnfQJ0p8Fq5llhih7EBWCjgR5jrP/34S1OPwmSpE+vBbarkxhgwtoHXEQFgsBDhgcVOoiAXqxaVNTV4QuCbMhKfCAf4EoxbETMh5h7OUDTNRj09ibmJu8UBVPVR6/Wj7hXGZZObbEcvJOs2j4DgfpQO27hfCZ11Q0uKD0xZfZU22f/6BWlT15YFSkNf1VqhJi+L5nY/CEOiK3rLom8gBpWzCYHYRRDduE91bfVCPFaXJyDs6KIaO6qTAhZu0ZVbEjB/iCiFzW6y3ESipmG6lY+l7eaPR0N/0h/OXWfYGw37s6nnz5aDfm86Xk6nk97Y8WazoTOyrOHM702G0547n4FtNnVn+Ks/XfYmLp94o/7EAfNvMuxvtN2qqRvGW9VEGVXiffpfryN6yeMzOmB4ACgYAAIok7ZxZVyRHw3y5ljYx8AlGYYt6XJxEl/L5F/ooAkWxALS+VtMNp7LSDUIsAKODSaoE5/ExkUCE30qAqzKXSbCJdX9e7q/C/76jsxvkUcEXheIGoycxNLSZrLErIQrKsXi5AacdNypuFVLxpv95vj1yemCfcALxods9hHLq3f2t/eai16iAvTT1fWuuR4i6LEuBVFA8TggmcX9FiyWhZX8k8PTnw5+PPiFinmkxNilYi0cU0LDohiAswwcYM9PV11sNNn75N2JPgR49SYFX5WGwZABDPnxF5C3wPiMWoQdRtkt+F/S+0EsiEnQ38JSzmqTMMkYJiEtAg268qqGYOoJjHY5CoZM5BBR2ofN++B9NcbSnoD2temmcmyVVZaya11v7oYRnWeNIjB8BsNFdlvSpF3zso8iL2C4KRWlKwTZoKuqy9a9DZ2O31R9+oeE8+kQcT4dlThvDHr5czVocrgjayAxqM7DBO8WJQxlD15m0s6JODcZno3zm4TKpY7LIgB4RPQ6F9CgDVMG0BWgRjy9kFeWInJUVSiorF6UM+RJYm3Y5ttqxdNDOimNPq+0W+yzQxVvPyPD0+PXKLaFQ+KwG3IeREo2oAv1bppkGb42gdR/nkmfDYsHMMdBJQ7jGaZo9/sTlapV5QSYHUXuW6Lv56QA0agXJko13uJa/OkDJpuvufGCL4sLkz0HAnAT7Cmq9Ox8rKWM3gkTroz26CWjYdgVoUms6yW1ZgKuwzDIwDSLscazq6LpESzkg8xP+JPRUzx6ZhVsbzbPP2oVpS90VKJY1mcEkzVC2witH+ECp8JK06ankTAFYGBt5lpLbVLQDTAblfgKkxQNNkNWi9wy97KIr7KONgkVQbROUmupTfJavIbhrusAX0RFSAHgcnMW+57uoXZRu644/IvBPgH2DsrgrVZSrGJTFJp++ewgU1FOVJ2qdBERgwfD5zf43gzCKFqsYZJcVaC474NBjEcRVFKI0k/d/yMjHfgtv4w4vYsiTACQTmnHbUeC3lDDwTtxqVbe4acwtQyjwiYxLK4lGNXbGEQhgkastEB7m8eSVjkRz8XFki1Nry2hSHLGjDK8bmqRV1lSNKIU13Qig5SPOgleSh4IpR4laBDgkhnlAb1TtkAqXlUizQp4qmH9lrsiWC2uNBEhhVkPJg1aF9p6LTBUyPEQiffMuRPmg4asMkFQq1UW1VBJyLvKPaMoe+isUF1RUJpQhknNKhJPuAOXIkgt9nOMQUQtCVtaUhhnJZjnpiAkv3XDwpNCl0LfLv5L/JIR6Q0xJSYxAwgyTJsuk4kdBHADHj5Ft9AUwgMABjLmn/cr9qgi4GU+1Kc8dJUOfZGo2iIhk78RHrUQLDgN2PEz3ZrJUOy7lyIHOqAoz3wwMWdbPSlKq8dYoyNymqibsmLZVSrNIBc04k4MWMBkRZkLK+/qJnFnUQOobAmypkxhk+B3szQ06XJknN8EdCUCzoRBpthBFoTFQQ0WGggHjud1qOwC1C++vQdTgwRn3O9Opv9CiyaCCWvXwctxGTjLwG9gomZW3e8z0KHK7YvCZOKbF6tvHBQL1vPW/bLtA9TXMFobWtaZIrMuFi8KkQZeLP795P3bhsf6f9aP2GM0R8mxPx9PammOdgbBU33IVOX82cmrk9cnZ+//DYvnnewqKwMUdF3eqvuFFGRTcvKImIZRSmeDp1xC31KWugaz9nNvTxReCyFqlzehjQ5Oi73WogCUrkFhbLcpf7S7sXYDv28cqIK5H3pSnWzsKQoQH9MT1VC94/5jVz3XVj3fPHBt1Vt6rq16S8/6qrEjq3HSqdJ6KFJROuY3iXipVikb7ugtWmgPcPA8r51wPcwlGHiCgm1/Dha9nslu598qO22CHY+v2qIJLnERqLlRp1jsvK45F4u2xDW91SBFO1Z6wbKmBrQdF6ItLqIlml4UcqOilZQS2aQfQ+7ndVFZ8AwVl1XHqLDa8LxYXnBtuzwIjXfvT77/4dUr+9nx2fO/NWrKcS9HZcJgVWr/WhK8Ohk2DjAGeDMT/1VRmE4jE3wGJKoULMYPwfAJi0xsHVQfnX9hRaww5YoGAP1sgKIpDmhz8u0RogpGalMBOwSbX0YIkG5SUzdgoZKtYg2gskTw8VCYOVTIhBW/OBeTnqlQV6jeG9CQDdBPxVCnCpto6e6byonAF5SlwJZree8mavcZYlZz5+WdWB3b+EMY6L/7foaodKJbH/1e/1ER0/XLCC05goY0ongxONlkybace9k+kO2NRBe6A2hxMSy8NWRhrv7ReQrDobH+PhgVyAQoVN6Zom5FxJbVRY2Xn1gpuHOpJ6OmvbVoZfW+NPBn7bQIefZEsusalow/C9tnwW9Xqcn+LFJ58scy8e4WS2QEWNBT9rkFww8umyz44Xxs9tGGHwHtRu25nEahy6vkAh3wWlZA3RKonBOZbsHMBihbKlvC38RoHWtnfbFYIA3MGrMjEDfrEqOMcyOEI+Z/Mla+fWsyI2Zd1kdmxnTVYXPQlZVGGYXPjGajOjBmexNOtqGJChH0/J63ytNNfal4376NN7TDyil1jRdQWjvgDvgqa2mt5cG0eyvX9o3kP1A1VHZn3WTqmwrcV+sCYxYJ2ngNAvkJyJtPDLQrTRHDMddZ/vcQqHWgTqTWDhWhNjYLYm1s3kCwzf0rom3soxNuY6eSeK09WlEuD4S2ZEukQbQrVZ83z6jnG7V3FuZbcqRbgeHdCdC8VO4Kjg9IoCRDzZQUGUXxE19FXUFXciqsiLcCRJGRUDCRahLf8yKTms6he7fK0a58yq3gpL9ZOXVYTM9+mtVu4aLfdyiTvduh5Ti1vFCDIguguHfdJMX4DiyOfF+tvrqdn1VE3/4028LSj2Drx/IiHVXf/pQ93CcDO5o/1O9RvL2Re8vrc5j5/J8j4OqB9ppc29619xCutUMHH5T2+/9HHSGvH+jU/xrKlVbfNrr+gepl7aVJujm43yIT17yKVnCaW/mBFNZHrGoRwSvx2NBXat1sUkuVL6nBaWEQWXgGOk5vDGKtrWWG+51tPzHQy0F2kd8gkyF56mCg2dQj3md06S/TXD6KgzaA4SunMrZLMTDoPpn8i7giSJJ2V7x+Q7Sir5hRgkVMCz6jMBVHk545wBdmj3sz/PIYM7+5ybq9gvFF9JNEgJF2+00mw5nkhlmiJvMudktL8wajsaKv2YCGDjs5hApFGM7HUcKpu+F42yr+VPAC49kqSooo8TCt04An3v9Qxl4jzBZTjC+jyk66lnnjBOtl3g8a3w2rDXe3Z2T4dnCTLenNV3vSaWnrvj5JlNEslgxM21FmCDgbxmr8nX3IcgzVEBz9mf7SM6tHxWtZm43dcoaFMX/E4vZ3oymvTA9AwwooVvQHOV79lqL50/I+jMrPaBH7WDj+GKBRyj8SLxX3MOTfAKYccCozstgJBR5qMQSqoJJh9zIsH2SYkmpAE7eoxdmTeaiMp+L2CV7wwVdTSvumDOvDGUV5s/biwTpKaxzRa0HLdr7b/0qe2/8D+a2F1xosKFdsuSF3UqONSVUAYT3usNOt99U4t02KNXVQ03xQxbFVaJi22siICak6G5pzFKrD/mNjJyI1WGkns9lFJf5WLW1lPLHFPtC5x3xcFad8h2nbm/hsvPkIjuMazqsmLOqg+lK665zh9Uh1KxvvKx/k9tU1/HNcPRe0CZjvb4Z1URxguuSAMiXbgYlVieXFwIO4Bvpw3MOd/wamjPXIV2MAAA=="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave121")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave121-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave121.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave121.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1",
                "GLCUDA_TELEMETRY": "1"}

    phase = "lmhead-ab"
    records = []
    orders = [("retained", "candidate"), ("candidate", "retained")] * 3
    for repeat, order in enumerate(orders):
        for arm in order:
            env = dict(prod_env)
            if arm == "candidate":
                env["GLCUDA_LMHEAD_GEMM"] = "1"
            measured = run([exe, MODEL, "profile"], cwd=TREE, env=env, check=False)
            save(f"profile-{repeat}-{arm}.log", measured)
            if measured.returncode or "[wave120-profile]" not in measured.stdout:
                raise RuntimeError(f"{arm} profile failed at repeat {repeat}")
            profile = json.loads(re.search(r"\[wave120-profile\]\s*(\{[^\n]+\})", measured.stdout).group(1))
            stages = [json.loads(x) for x in re.findall(r"\[wave120-stage\]\s*(\{[^\n]+\})", measured.stdout)]
            lm = next(x for x in stages if x["name"] == "lm_head")
            if len(stages) != 9 or profile["oracle_token"] != 3323:
                raise RuntimeError(f"{arm} contract failed: {profile}, {len(stages)} stages")
            records.append({"repeat": repeat, "arm": arm, **profile,
                            "lm_head_ms": lm["total_ms"]})
    by_arm = {arm: [x for x in records if x["arm"] == arm]
              for arm in ("retained", "candidate")}
    med = lambda arm, key: statistics.median(x[key] for x in by_arm[arm])
    summary = {"wave": 121, "gpu": fields, "model": model_meta,
               "records": records,
               "median_gpu_prefill_ms": {arm: med(arm, "gpu_prefill_ms") for arm in by_arm},
               "median_prefill_tps": {arm: med(arm, "gpu_prefill_tps") for arm in by_arm},
               "median_lm_head_ms": {arm: med(arm, "lm_head_ms") for arm in by_arm}}
    summary["speedup"] = (summary["median_gpu_prefill_ms"]["retained"] /
                          summary["median_gpu_prefill_ms"]["candidate"])
    summary["lm_head_speedup"] = (summary["median_lm_head_ms"]["retained"] /
                                  summary["median_lm_head_ms"]["candidate"])
    summary["target_15000_tps_achieved"] = summary["median_prefill_tps"]["candidate"] >= 15000
    (RESULTS / "wave121-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE121_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
